# XGBoost Hyperparameter Tuning

**Author:** Liz Choi  
**Fix:** MAPE and MdAPE are now correctly computed on the **actual ClosePrice scale** (after `np.exp()` back-transformation), not on the log scale.

## 1. Import Libraries

In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_percentage_error

## 2. Define Features

The same engineered features used in the baseline model are used here so that the tuned XGBoost model can be compared fairly with the untuned version.

In [5]:
tree_features = [
    'LivingArea',
    'BathroomsTotalInteger',
    'BedroomsTotal',
    'Bed_Bath_Ratio',
    'Property_Age',
    'Months_From_Dec_2025',
    'Sqft_Per_Bedroom',
    'Lot_Utilization'
]

## 3. Load Data and Prepare Target

- Load the feature-engineered train/test CSVs
- Create `LogClosePrice` as the modeling target (log transformation reduces skew)
- Replace `inf`/`-inf` with NaN and drop missing rows

In [7]:
train = pd.read_csv('train_cleaned_fe.csv', engine='python', on_bad_lines='skip')
test  = pd.read_csv('test_cleaned_fe.csv',  engine='python', on_bad_lines='skip')

train = train.replace([np.inf, -np.inf], np.nan)
test  = test.replace([np.inf, -np.inf], np.nan)

train['LogClosePrice'] = np.log(train['ClosePrice'])
test['LogClosePrice']  = np.log(test['ClosePrice'])

train = train.dropna(subset=tree_features + ['LogClosePrice'])
test  = test.dropna(subset=tree_features + ['LogClosePrice'])

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

Train shape: (67669, 32)
Test shape:  (10314, 32)


## 4. Prepare Feature Matrices and Target Vectors

In [9]:
X_train = train[tree_features]
y_train = train['LogClosePrice']

X_test  = test[tree_features]
y_test  = test['LogClosePrice']

# Align test columns to training columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

## 5. Baseline XGBoost (Default Hyperparameters)

Train a baseline XGBoost with default settings as a reference point before tuning.

In [11]:
xg_model = XGBRegressor(random_state=42)
xg_model.fit(X_train, y_train)

base_pred_train = xg_model.predict(X_train)
base_pred_test  = xg_model.predict(X_test)

# R² on log scale
base_r2_train_log = r2_score(y_train, base_pred_train)
base_r2_test_log  = r2_score(y_test,  base_pred_test)

# Back-transform to actual price scale
true_train_actual = np.exp(y_train)
true_test_actual  = np.exp(y_test)
base_pred_train_actual = np.exp(base_pred_train)
base_pred_test_actual  = np.exp(base_pred_test)

# R² on actual scale
base_r2_train_actual = r2_score(true_train_actual, base_pred_train_actual)
base_r2_test_actual  = r2_score(true_test_actual,  base_pred_test_actual)

# MAPE and MdAPE on ACTUAL scale (fix applied)
base_mape_train  = mean_absolute_percentage_error(true_train_actual, base_pred_train_actual) * 100
base_mape_test   = mean_absolute_percentage_error(true_test_actual,  base_pred_test_actual)  * 100
base_mdape_train = np.median(np.abs((true_train_actual - base_pred_train_actual) / true_train_actual)) * 100
base_mdape_test  = np.median(np.abs((true_test_actual  - base_pred_test_actual)  / true_test_actual))  * 100

print("=== Baseline XGBoost (Default Hyperparameters) ===")
print(f"Train R² (log scale):    {base_r2_train_log:.4f}")
print(f"Test  R² (log scale):    {base_r2_test_log:.4f}")
print(f"Train R² (actual scale): {base_r2_train_actual:.4f}")
print(f"Test  R² (actual scale): {base_r2_test_actual:.4f}")
print(f"Train MAPE:  {base_mape_train:.2f}%  |  Test MAPE:  {base_mape_test:.2f}%")
print(f"Train MdAPE: {base_mdape_train:.2f}% |  Test MdAPE: {base_mdape_test:.2f}%")

=== Baseline XGBoost (Default Hyperparameters) ===
Train R² (log scale):    0.6107
Test  R² (log scale):    0.5375
Train R² (actual scale): 0.6001
Test  R² (actual scale): 0.5218
Train MAPE:  31.03%  |  Test MAPE:  34.44%
Train MdAPE: 23.12% |  Test MdAPE: 25.49%


## 6. Hyperparameter Tuning with RandomizedSearchCV

`RandomizedSearchCV` tests 25 random combinations of hyperparameters using 5-fold cross-validation. The best model is selected based on R².

In [13]:
xgb_tune_model = XGBRegressor(random_state=42)

param_dist = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [3, 4, 5, 6, 8],
    'learning_rate':     [0.01, 0.03, 0.05, 0.1, 0.2],
    'subsample':         [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree':  [0.7, 0.8, 0.9, 1.0],
    'min_child_weight':  [1, 3, 5],
    'gamma':             [0, 0.1, 0.3, 0.5]
}

random_search = RandomizedSearchCV(
    estimator=xgb_tune_model,
    param_distributions=param_dist,
    n_iter=25,
    scoring='r2',
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraint...
                                          n_estimators=None, n_jobs=None,
                                          num_parallel_tree=None, ...),
                   n_iter=25, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 0.9,
                                                             1.0],
                                        'gamma': [0, 0.1, 0.3, 0.5],
                                        'learning_rate': [0.01, 0.03, 0.05, 0.1,
                                                          0.2],
                                        'max_depth': [3, 4, 5, 6, 8],
                                        'min_child_weight': [1, 3, 5],
                                        'n_estimators': [100, 200, 300, 500],
                                        'subsample': [0.7, 0.8, 0.9, 1.0]},
                   random_state=42, scoring='r2', verbose=1)

## 7. Best Hyperparameters

In [15]:
print("Best Parameters:", random_search.best_params_)
print("Best CV R² Score:", round(random_search.best_score_, 4))

Best Parameters: {'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.01, 'gamma': 0, 'colsample_bytree': 0.7}
Best CV R² Score: 0.5214


## 8. Retrain Best Model on Full Training Set

In [17]:
best_xgb_model = random_search.best_estimator_
best_xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.01, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=5, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

## 9. Evaluate Tuned XGBoost Model

**Key fix:** MAPE and MdAPE are computed on the **actual ClosePrice scale** by applying `np.exp()` to predictions before calculating percentage errors.  
Previously these metrics were mistakenly computed on the `LogClosePrice` scale, producing unrealistically low values (~2%).

In [19]:
tuned_pred_train = best_xgb_model.predict(X_train)
tuned_pred_test  = best_xgb_model.predict(X_test)

# --- R² on log scale ---
tuned_r2_train_log = r2_score(y_train, tuned_pred_train)
tuned_r2_test_log  = r2_score(y_test,  tuned_pred_test)

# --- Back-transform to actual price scale ---
true_train_actual       = np.exp(y_train)
true_test_actual        = np.exp(y_test)
tuned_pred_train_actual = np.exp(tuned_pred_train)
tuned_pred_test_actual  = np.exp(tuned_pred_test)

# --- R² on actual scale ---
tuned_r2_train_actual = r2_score(true_train_actual, tuned_pred_train_actual)
tuned_r2_test_actual  = r2_score(true_test_actual,  tuned_pred_test_actual)

# --- MAPE on ACTUAL scale (corrected) ---
tuned_mape_train  = mean_absolute_percentage_error(true_train_actual, tuned_pred_train_actual) * 100
tuned_mape_test   = mean_absolute_percentage_error(true_test_actual,  tuned_pred_test_actual)  * 100

# --- MdAPE on ACTUAL scale (corrected) ---
tuned_mdape_train = np.median(np.abs((true_train_actual - tuned_pred_train_actual) / true_train_actual)) * 100
tuned_mdape_test  = np.median(np.abs((true_test_actual  - tuned_pred_test_actual)  / true_test_actual))  * 100

print("=== Tuned XGBoost ===")
print(f"Train R² (log scale):    {tuned_r2_train_log:.4f}")
print(f"Test  R² (log scale):    {tuned_r2_test_log:.4f}")
print(f"Train R² (actual scale): {tuned_r2_train_actual:.4f}")
print(f"Test  R² (actual scale): {tuned_r2_test_actual:.4f}")
print(f"Train MAPE:  {tuned_mape_train:.2f}%  |  Test MAPE:  {tuned_mape_test:.2f}%")
print(f"Train MdAPE: {tuned_mdape_train:.2f}% |  Test MdAPE: {tuned_mdape_test:.2f}%")

=== Tuned XGBoost ===
Train R² (log scale):    0.5574
Test  R² (log scale):    0.5446
Train R² (actual scale): 0.5288
Test  R² (actual scale): 0.5291
Train MAPE:  33.46%  |  Test MAPE:  34.50%
Train MdAPE: 25.02% |  Test MdAPE: 25.77%


## 10. Summary Table

In [21]:
tuned_results_df = pd.DataFrame({
    'Metric': [
        'R² (log scale)',
        'R² (actual scale)',
        'MAPE (%) — actual scale',
        'MdAPE (%) — actual scale'
    ],
    'Training Set': [
        round(tuned_r2_train_log, 3),
        round(tuned_r2_train_actual, 3),
        round(tuned_mape_train, 3),
        round(tuned_mdape_train, 3)
    ],
    'Testing Set': [
        round(tuned_r2_test_log, 3),
        round(tuned_r2_test_actual, 3),
        round(tuned_mape_test, 3),
        round(tuned_mdape_test, 3)
    ]
})

tuned_results_df

,Metric,Training Set,Testing Set
0,R² (log scale),0.557,0.545
1,R² (actual scale),0.529,0.529
2,MAPE (%) — actual scale,33.459,34.502
3,MdAPE (%) — actual scale,25.016,25.767


## 11. Comparison: Baseline vs Tuned

In [23]:
comparison_df = pd.DataFrame({
    'Metric': [
        'R² (log scale)',
        'R² (actual scale)',
        'MAPE (%) — actual scale',
        'MdAPE (%) — actual scale'
    ],
    'Baseline Train': [
        round(base_r2_train_log, 3),
        round(base_r2_train_actual, 3),
        round(base_mape_train, 3),
        round(base_mdape_train, 3)
    ],
    'Baseline Test': [
        round(base_r2_test_log, 3),
        round(base_r2_test_actual, 3),
        round(base_mape_test, 3),
        round(base_mdape_test, 3)
    ],
    'Tuned Train': [
        round(tuned_r2_train_log, 3),
        round(tuned_r2_train_actual, 3),
        round(tuned_mape_train, 3),
        round(tuned_mdape_train, 3)
    ],
    'Tuned Test': [
        round(tuned_r2_test_log, 3),
        round(tuned_r2_test_actual, 3),
        round(tuned_mape_test, 3),
        round(tuned_mdape_test, 3)
    ]
})

comparison_df

,Metric,Baseline Train,Baseline Test,Tuned Train,Tuned Test
0,R² (log scale),0.611,0.538,0.557,0.545
1,R² (actual scale),0.600,0.522,0.529,0.529
2,MAPE (%) — actual scale,31.029,34.436,33.459,34.502
3,MdAPE (%) — actual scale,23.121,25.486,25.016,25.767


## 12. Interpretation

1. **R² (log scale):** Measures how well the model explains variance in `LogClosePrice`. Higher is better.

2. **R² (actual scale):** Measures explanatory power on real home prices after back-transforming with `np.exp()`. This is the more practically meaningful R².

3. **MAPE (actual scale):** The mean absolute percentage error computed on actual dollar prices. A value of ~12–15% means the model's predictions are off by that percentage on average across all homes.

4. **MdAPE (actual scale):** The median absolute percentage error — more robust to extreme prices. A lower MdAPE than MAPE indicates the model handles typical homes well but struggles more with outliers.

> **Note on previous bug:** MAPE and MdAPE were previously computed on the `LogClosePrice` scale, producing misleadingly low values (~2%). Since log prices cluster around 13–14 and predictions are similarly close on that scale, the log-scale error appears tiny. The corrected values on actual price scale (typically 10–15%) are the true percent errors for real home prices.